# SmartRecipe AI: Baseline Model Validation and Ranking Evaluation

This notebook validates the **existing saved baseline artifacts** and evaluates the TF-IDF + combined-embedding retrieval model. It complements, rather than replaces, `data_preprocessing.ipynb`, `EDA.ipynb`, and `advanced_modelling.ipynb`.

**Reproducibility note:** EDA and preprocessing notebooks document data lineage and preparation; the actual runtime baseline used by the app loads `input/sampled_data.pkl`, `input/tfidf_vectorizer.pkl`, and `input/combined_embeddings.pkl`. This notebook tests those exact artifacts without fitting a new model. Human relevance labels are not fabricated. Recall/MAP are only calculated if an independently judged relevance pool is available.


In [ ]:
from pathlib import Path
import json, time, sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend" / "baseline_service.py").exists():
    # Handle launching the notebook from the parent folder in Jupyter
    nested = PROJECT_ROOT / "RecipeRecommendation_Project"
    if (nested / "backend" / "baseline_service.py").exists():
        PROJECT_ROOT = nested
        import os
        os.chdir(PROJECT_ROOT)

assert (PROJECT_ROOT / "backend" / "baseline_service.py").exists(), (
    "Set the notebook working directory to the folder containing backend/, input/, and data/."
)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


In [ ]:
# Validate the three saved artifacts required by the runtime baseline.
from backend.config import SAMPLED_DATA_PATH, TFIDF_VECTORIZER_PATH, COMBINED_EMBEDDINGS_PATH

artifacts = {
    "sampled recipe data": SAMPLED_DATA_PATH,
    "saved TF-IDF vectorizer": TFIDF_VECTORIZER_PATH,
    "combined embedding matrix": COMBINED_EMBEDDINGS_PATH,
}
artifact_rows = []
for label, path in artifacts.items():
    artifact_rows.append({"artifact": label, "path": str(path), "exists": path.exists(),
                          "size_MB": round(path.stat().st_size / (1024**2), 2) if path.exists() else None})
artifact_df = pd.DataFrame(artifact_rows)
display(artifact_df)
if not artifact_df["exists"].all():
    raise FileNotFoundError("One or more saved baseline artifacts are missing. See table above.")


In [ ]:
# Load the exact baseline service used by the application.
from backend.baseline_service import baseline_service
from backend.input_processor import parse_requirements
from backend.preference_ranker import filter_candidates_by_constraints, score_candidates, ingredient_match_lists, _diet_violations


In [ ]:
baseline_service.load_models()
print("Baseline artifacts loaded.")
print("Recipe rows:", len(baseline_service._sampled_data))
feature_count = (len(baseline_service._vectorizer.get_feature_names_out())
                 if hasattr(baseline_service._vectorizer, "get_feature_names_out")
                 else len(baseline_service._vectorizer.get_feature_names()))
print("TF-IDF feature count:", feature_count)
print("Combined embedding shape:", baseline_service._combined_embeddings.shape)


In [ ]:
# Load the fixed nine assessment queries.
TEST_CASES_PATH = PROJECT_ROOT / "data" / "assessment_test_cases.json"
test_cases = json.loads(TEST_CASES_PATH.read_text(encoding="utf-8"))
print("Test cases:", len(test_cases))
pd.DataFrame(test_cases)[["test_id", "request", "purpose"]]


In [ ]:
# Run baseline retrieval only. This does not call Gemini.
# Increase TOP_K only if you deliberately want a wider candidate pool.
TOP_K = 5
retrieval_rows = []
for test in test_cases:
    started = time.perf_counter()
    candidates = baseline_service.get_candidates(test["request"], num_candidates=TOP_K)
    elapsed = time.perf_counter() - started
    req = parse_requirements(test["request"])
    valid, rejected = filter_candidates_by_constraints(candidates, req)
    for candidate in candidates:
        matched, missing = ingredient_match_lists(candidate, req)
        diet_violations = _diet_violations(candidate, req)
        time_ok = (not req.max_cooking_minutes or candidate.minutes <= 0 or candidate.minutes <= req.max_cooking_minutes)
        hard_ok = (not missing) and (not diet_violations) and time_ok
        retrieval_rows.append({
            "test_id": test["test_id"], "query": test["request"],
            "rank": candidate.baseline_rank, "recipe_name": candidate.name,
            "similarity_score": candidate.baseline_similarity_score,
            "minutes": candidate.minutes, "ingredients": ", ".join(candidate.ingredients),
            "tags": ", ".join(candidate.tags), "matched_required_ingredients": ", ".join(matched),
            "missing_required_ingredients": ", ".join(missing),
            "diet_violations": "; ".join(diet_violations),
            "time_compliant": bool(time_ok), "hard_constraints_passed": int(hard_ok),
            "retrieval_latency_seconds": round(elapsed, 6),
        })
baseline_results = pd.DataFrame(retrieval_rows)
display(baseline_results.head(15))


In [ ]:
# Summary of actual retrieval and constraint results.
summary = baseline_results.groupby("test_id").agg(
    candidates_returned=("recipe_name", "count"),
    top_similarity=("similarity_score", "max"),
    hard_constraint_passes=("hard_constraints_passed", "sum"),
    time_compliant_candidates=("time_compliant", "sum"),
    retrieval_latency_seconds=("retrieval_latency_seconds", "first"),
).reset_index()
summary["hard_constraint_pass_rate_top5_pct"] = (
    100 * summary["hard_constraint_passes"] / summary["candidates_returned"].replace(0, np.nan)
).round(2)
display(summary)
print("Top-5 candidate-level hard-constraint pass rate (%):",
      round(baseline_results["hard_constraints_passed"].mean() * 100, 2))
print("Mean retrieval latency (s):", round(summary["retrieval_latency_seconds"].mean(), 4))


In [ ]:
# Export actual retrieved candidates for transparent human relevance judgments.
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
candidate_export = OUTPUT_DIR / "baseline_candidates_for_judgment.csv"
baseline_results.to_csv(candidate_export, index=False)
print("Saved candidate export:", candidate_export)
print("Review each recipe against its query before assigning relevance labels.")


## Ranking metrics from human judgments

The next cell uses `outputs/baseline_relevance_judgments.csv` if present. Precision@5, MRR@5, and binary NDCG@5 can be computed from judged top-five results. **Recall@5 and MAP@5 are not computed from only the same top-five items**, because that would make the denominator incomplete. They require total relevant items from a broader, independently judged pool. Do not fill labels automatically or fabricate them.


In [ ]:
judgment_path = OUTPUT_DIR / "baseline_relevance_judgments.csv"
if not judgment_path.exists():
    print("No human judgment file found yet. Use baseline_candidates_for_judgment.csv to review results.")
else:
    judgments = pd.read_csv(judgment_path)
    required_cols = {"test_id", "rank", "recipe_name", "relevance"}
    if not required_cols.issubset(judgments.columns):
        print("Judgment file is missing required columns:", required_cols - set(judgments.columns))
    elif judgments["relevance"].isna().any() or (judgments["relevance"].astype(str).str.strip() == "").any():
        print("Human relevance labels are incomplete. Complete the relevance column with 1 or 0 first.")
    else:
        judgments["relevance"] = pd.to_numeric(judgments["relevance"], errors="raise").astype(int)
        if not judgments["relevance"].isin([0, 1]).all():
            raise ValueError("Relevance labels must be binary 0/1.")
        def _dcg(rel):
            rel = np.asarray(rel, dtype=float)
            return float(np.sum((2**rel - 1) / np.log2(np.arange(2, len(rel) + 2))))
        metric_rows = []
        for test_id, group in judgments.groupby("test_id"):
            group = group.sort_values("rank")
            rel = group["relevance"].tolist()[:5]
            p5 = sum(rel) / 5 if len(rel) == 5 else np.nan
            first = next((i for i, v in enumerate(rel, 1) if v == 1), None)
            ideal_dcg = _dcg(sorted(rel, reverse=True))
            metric_rows.append({"test_id": test_id, "Precision@5": p5,
                                "MRR@5": 1 / first if first else 0.0,
                                "NDCG@5": _dcg(rel) / ideal_dcg if ideal_dcg else np.nan,
                                "relevant_in_top5": sum(rel),
                                "Recall@5": np.nan, "MAP@5": np.nan})
        ranking_metrics = pd.DataFrame(metric_rows)
        display(ranking_metrics.round(3))
        display(ranking_metrics[["Precision@5", "MRR@5", "NDCG@5"]].mean().rename("mean_score").to_frame().round(3))
        print("Recall@5/MAP@5 remain blank until a broader judged relevance pool is available.")


In [ ]:
# Save all current baseline evaluation artifacts with a timestamp.
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
retrieval_path = OUTPUT_DIR / f"baseline_retrieval_candidates_{stamp}.csv"
summary_path = OUTPUT_DIR / f"baseline_retrieval_summary_{stamp}.csv"
baseline_results.to_csv(retrieval_path, index=False)
summary.to_csv(summary_path, index=False)
print("Saved:", retrieval_path.name)
print("Saved:", summary_path.name)


## Interpretation and limitations

- This notebook evaluates the **frozen saved retrieval artifacts**, not a newly trained classifier.
- Similarity scores are not accuracy probabilities.
- Relevance metrics depend on human judgments and the evaluation pool.
- Candidate-level hard-constraint pass rate is not the same as user-facing recommendation success rate.
- Allergy safety cannot be certified from ingredient text alone.
- Keep the original EDA/preprocessing/advanced modelling notebooks as model-development lineage. Cite or describe their outputs only if you actually ran and inspected them; the live app's baseline provenance is the saved artifacts validated above.
